# 📄 PDF-based RAG System (100% Open-Source / Free)

**Stack used — every piece below is free, open-source, and needs NO API key, NO login/gated access, and NO paid credits:**

| Component | Library / Model | License | Notes |
|---|---|---|---|
| PDF text extraction | `pypdf` | BSD-3 | Pure Python, no external service |
| Text chunking | custom (pure Python) | - | No dependency needed |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` | Apache-2.0 | Runs fully locally, downloaded once from Hugging Face Hub, no key required |
| Vector store | `faiss-cpu` (Meta AI) | MIT | Local, in-memory, no server |
| LLM (generation) | `TinyLlama/TinyLlama-1.1B-Chat-v1.0` | Apache-2.0 | NOT gated — anyone can download without requesting access, unlike Llama/Mistral/Gemma |
| Optional UI | `gradio` | Apache-2.0 | Runs inline in Colab |

> ✅ I deliberately avoided anything that requires an API key (OpenAI, Cohere, Anthropic), anything gated on Hugging Face (Llama-2/3, Mistral, Gemma need you to request+accept a license), and anything with a restrictive license (e.g. PyMuPDF is AGPL — swapped for `pypdf` which is BSD and safe for any use).

**How to use this notebook:** Runtime → Change runtime type → GPU (T4) is recommended but not required (TinyLlama runs on CPU too, just slower). Run cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.0 MB/s eta 0:00:00


## 2. Upload your PDF

In [ ]:
from google.colab import files

uploaded = files.upload()  # choose your PDF file
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}")

Saving synopsis E-7.pdf to synopsis E-7.pdf
Uploaded: synopsis E-7.pdf


## 3. Extract text from the PDF (`pypdf`)

In [ ]:
from pypdf import PdfReader

def extract_pdf_text(path):
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append({"page_num": i + 1, "text": text})
    return pages

pages = extract_pdf_text(pdf_path)
print(f"Extracted {len(pages)} pages")
print(pages[0]["text"][:500])

Extracted 6 pages
   
 
   
DEPARTMENT OF COMPUTER SCIENCE &  ENGINEERING      
THE NATIONAL INSTITUTE OF ENGINEERING 
(An Autonomous Institution) 
 
Project Synopsis 
 
Batch No : E7    Date : 12/08/2026 
USN Name Email-ID Mobile Number 
4NI23CI012 Amogh S Acharya 2023ci_amoghsacharya_a@nie.a
c.in 
9845901716 
4NI23CI030 Darshan K 2023ci_darshank_a@nie.ac.in 8310082026 
4NI23CI038 Gireesh Sadanand Patgar 2023ci_gireeshsadanandpatgar_a
@nie.ac.in 
8861661013 
4NI23CI060 Naveen 2023ci_naveen_a@nie.ac.in 9380710847


## 4. Chunk the text
We split into overlapping chunks so context isn't lost at boundaries. Each chunk keeps track of which page it came from (for citations).

In [ ]:
def chunk_text(pages, chunk_size=800, overlap=150):
    """chunk_size / overlap are in characters (simple, dependency-free, robust)."""
    chunks = []
    for p in pages:
        text = p["text"].strip()
        if not text:
            continue
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            chunks.append({"page_num": p["page_num"], "text": chunk})
            start += chunk_size - overlap
    return chunks

chunks = chunk_text(pages)
print(f"Created {len(chunks)} chunks")
print(chunks[0])

Created 19 chunks
{'page_num': 1, 'text': 'DEPARTMENT OF COMPUTER SCIENCE &  ENGINEERING      \nTHE NATIONAL INSTITUTE OF ENGINEERING \n(An Autonomous Institution) \n \nProject Synopsis \n \nBatch No : E7    Date : 12/08/2026 \nUSN Name Email-ID Mobile Number \n4NI23CI012 Amogh S Acharya 2023ci_amoghsacharya_a@nie.a\nc.in \n9845901716 \n4NI23CI030 Darshan K 2023ci_darshank_a@nie.ac.in 8310082026 \n4NI23CI038 Gireesh Sadanand Patgar 2023ci_gireeshsadanandpatgar_a\n@nie.ac.in \n8861661013 \n4NI23CI060 Naveen 2023ci_naveen_a@nie.ac.in 9380710847 \nProject Title: AI & ML Enabled IoT-Based Precision Agriculture Platform for Smart Irrigation and Crop Disease \nDetection \nIntroduction:    \nAgriculture faces challenges such as improper irrigation, delayed crop disease detection, uncertain weather conditions, and \ninefficient use of water resources, which r'}


## 5. Generate embeddings + build a FAISS index
`all-MiniLM-L6-v2` is a small (~80MB), fast, high-quality open-source embedding model — no key needed, downloads once from Hugging Face.

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # so we can use inner product == cosine similarity
)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # cosine similarity via normalized inner product
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index built with 19 vectors of dimension 384


## 6. Retrieval function

In [ ]:
def retrieve(query, top_k=4):
    query_vec = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "text": chunks[idx]["text"],
            "page_num": chunks[idx]["page_num"],
            "score": float(score),
        })
    return results

# quick sanity check
test_results = retrieve("What is this document about?")
for r in test_results:
    print(f"[page {r['page_num']}, score {r['score']:.3f}] {r['text'][:120]}...\n")

[page 3, score 0.168] he proposed...

[page 2, score 0.134] • To provide irrigation history, smart notifications, and water usage analytics.  
• To reduce water wastage, manual eff...

[page 6, score 0.111] secondary connection. 
Advantages 
• Automatic irrigation  
• Real-time monitoring  
• Early disease detection  
• Crop ...

[page 5, score 0.103] roduction The system encourages efficient use 
of agricultural resources. Data-driven 
irrigation helps avoid over-water...



## 7. Load the open-source LLM

`TinyLlama-1.1B-Chat-v1.0` is used here because it is:
- **Fully open**, Apache-2.0 licensed (safe for academic/commercial use)
- **Not gated** — no "request access" step on Hugging Face, unlike Llama-3, Mistral, or Gemma
- **Small (1.1B params)** — runs comfortably on Colab's free T4 GPU or even CPU

> Want better answer quality and have a GPU? Swap the `model_id` below for `"microsoft/Phi-3-mini-4k-instruct"` (MIT license, also ungated, ~3.8B params, noticeably smarter but slower/heavier).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.3,
    do_sample=False,
    repetition_penalty=1.1,
)

print("LLM loaded.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded.


## 8. The RAG pipeline: retrieve context → build prompt → generate grounded answer

In [ ]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Page {c['page_num']}] {c['text']}" for c in retrieved_chunks
    )
    system_msg = (
        "You are a helpful assistant that answers questions using ONLY the "
        "provided context from a PDF document. If the answer is not in the "
        "context, say you don't know. Always mention the page number(s) you "
        "used when relevant."
    )
    user_msg = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return prompt

def rag_answer(query, top_k=4):
    retrieved = retrieve(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    output = llm_pipeline(prompt)[0]["generated_text"]
    # strip the prompt portion off, keep only the newly generated answer
    answer = output[len(prompt):].strip()
    sources = sorted(set(c["page_num"] for c in retrieved))
    return answer, sources, retrieved

# --- Try it ---
question = "Summarize the main topic of this document."
answer, sources, retrieved = rag_answer(question)
print("Q:", question)
print("\nA:", answer)
print("\nSources: pages", sources)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Q: Summarize the main topic of this document.

A: The main topic of this document is an IoT-powered smart agricultural system that includes features such as automated irrigation, real-time monitoring, early disease detection, crop growth tracking, water and cost savings, remote farm monitoring, scalable IoT-AI platform, and integration with weather APIs, soil moisture sensors, temperature and humidity sensors, rain sensors, water level sensors, and relay modules. The document provides details on the hardware requirements, software requirements, and guide details for implementation. The proposed system is designed to help farmers optimize their crops' growth and productivity while reducing manual labor and improving crop quality.

Sources: pages [2, 3, 4, 6]


## 9. Interactive Q&A loop (run and type your questions in the box)

In [ ]:
while True:
    q = input("\nAsk a question about the PDF (or type 'exit'): ")
    if q.strip().lower() == "exit":
        break
    ans, srcs, _ = rag_answer(q)
    print(f"\nAnswer: {ans}\n(Sources: pages {srcs})")


Ask a question about the PDF (or type 'exit'): What is the project?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: The project is a smart irrigation system that uses IoT, AI, ML, cloud technologies, and a mobile application to monitor soil moisture, rainfall, water level, weather forecasts, irrigation history, crop health, and crop disease detection. The system aims to improve crop health, conserve water, and reduce manual intervention by providing real-time data and automated irrigation decisions based on sensor readings and weather forecasts. The project is designed to support sustainable agriculture by reducing wasteful water use and increasing crop yields. It is a scalable IoT-AI platform that can be used by farmers for remote monitoring, crop health
(Sources: pages [2, 5, 6])

Ask a question about the PDF (or type 'exit'): What are the hardware requirements?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: The hardware requirements for the AI & ML Enabled IoT-Based Precision Agriculture Platform for Smart Irrigation and Crop Disease Detection include:

1. ESP32 Development Board: This is a microcontroller board that can be used to develop and test the software components of the platform. It has a processor speed of up to 800MHz, 4MB RAM, and 8MB Flash memory.

2. Soil Moisture Sensor: This sensor measures the amount of moisture in the soil by measuring the electrical conductivity of the soil. It requires a power source and a connection cable.

3. DHT2
(Sources: pages [1, 2, 3])

Ask a question about the PDF (or type 'exit'): What are the software requirements?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: The software requirements include:

1. Arduino IDE
2. Arduino Studio
3. Python 3.11+
4. Visual Studio Code
5. Flask / Streamlit
6. HTML, CSS, JavaScript
7. TensorFlow / Keras
8. OpenCV
9. Firebase or Supabase
10. Git & GitHub
11. Weather API (OpenWeatherMap)
12. FastAPI (optional)
13. Chart.js / Recharts (dashboard graphs)

Note: These software requirements may vary depending on the specific project requirements.
(Sources: pages [1, 2])

Ask a question about the PDF (or type 'exit'): exit


## 10. (Optional) Simple Gradio web UI
Gives you a shareable chat-style interface, still 100% local/free — `gradio` just renders the UI, no external calls are made.

In [ ]:
import gradio as gr

def gradio_chat(message, history):
    answer, sources, _ = rag_answer(message)
    return f"{answer}\n\n📄 Sources: page(s) {sources}"

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="PDF RAG Assistant (Open-Source Stack)",
    description="Ask questions about your uploaded PDF. Powered by MiniLM embeddings + FAISS + TinyLlama.",
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f5e8393a6893e5e7d9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Notes for your GenAI submission

- **Why these choices are safe/free**: `pypdf`, `sentence-transformers`, `faiss-cpu`, `transformers`, and `gradio` are all permissively licensed (BSD/Apache/MIT) open-source PyPI packages — `pip install` only, no signup. `all-MiniLM-L6-v2` and `TinyLlama-1.1B-Chat-v1.0` are open-weight models on the Hugging Face Hub with **no gating/access-request step** and Apache-2.0 licenses, so they download and run with zero configuration and no API key.
- **Architecture recap**: PDF → text extraction (`pypdf`) → chunking → embeddings (`MiniLM`) → vector index (`FAISS`) → similarity search → prompt construction → local LLM generation (`TinyLlama`) → grounded answer with page citations.
- **To improve quality further** (optional, still free): try a larger ungated model such as `microsoft/Phi-3-mini-4k-instruct` or `Qwen/Qwen2.5-1.5B-Instruct`; increase `top_k`; use semantic/sentence-aware chunking instead of fixed character windows; add a re-ranker like `cross-encoder/ms-marco-MiniLM-L-6-v2` (also open-source, free).
- **Multiple PDFs**: to extend this to a folder of PDFs, just loop `extract_pdf_text` over each file, tag chunks with the filename as well as the page number, and build one shared FAISS index.

In [ ]:
evaluation_data = [
    {
        "question": "What is the main project proposed in the synopsis?",
        "expected_answer": "An AI and ML enabled IoT-based precision agriculture platform for smart irrigation and crop disease detection.",
        "type": "direct"
    },
    {
        "question": "Which microcontroller is used in the proposed system?",
        "expected_answer": "ESP32.",
        "type": "direct"
    },
    {
        "question": "What sensors are used to monitor farm conditions?",
        "expected_answer": "Soil moisture, temperature, humidity, rainfall, and water level sensors.",
        "type": "direct"
    },
    {
        "question": "What is the purpose of the smart irrigation system?",
        "expected_answer": "To automatically manage irrigation based on sensor readings and weather forecast information and optimize water usage.",
        "type": "direct"
    },
    {
        "question": "What does the AI-based crop disease detection module analyze?",
        "expected_answer": "Crop leaf images.",
        "type": "direct"
    },
    {
        "question": "Which technology is used for crop disease detection?",
        "expected_answer": "AI/ML and image processing techniques, with deep learning used for analyzing crop leaf images.",
        "type": "direct"
    },
    {
        "question": "What software frameworks are listed in the project requirements?",
        "expected_answer": "Flask/Streamlit, TensorFlow/Keras, OpenCV, Firebase or Supabase, and other listed software tools.",
        "type": "direct"
    },
    {
        "question": "What hardware is required for the proposed system?",
        "expected_answer": "ESP32 development board, soil moisture sensor, DHT22 temperature and humidity sensor, rain sensor, water level sensor, relay module, mini water pump, breadboard, jumper wires, power supply, and smartphone camera.",
        "type": "direct"
    },
    {
        "question": "What are the main advantages of the proposed system?",
        "expected_answer": "Automatic irrigation, real-time monitoring, early disease detection, crop growth tracking, water and cost savings, reduced manual effort, remote farm monitoring, and a scalable IoT-AI platform.",
        "type": "direct"
    },
    {
        "question": "Which weather API is mentioned in the software requirements?",
        "expected_answer": "OpenWeatherMap Weather API.",
        "type": "direct"
    }
]

In [ ]:
evaluation_data += [
    {
        "question": "How does the proposed system use IoT and weather information for irrigation?",
        "expected_answer": "The ESP32 collects soil moisture, temperature, humidity, rainfall, and water-level data, and the system uses these sensor readings together with weather forecast information to automatically manage irrigation.",
        "type": "multi_context"
    },
    {
        "question": "How does the proposed platform help farmers monitor their farms?",
        "expected_answer": "The application provides real-time sensor data, irrigation status, weather updates, notifications, and irrigation history.",
        "type": "multi_context"
    },
    {
        "question": "How does the project combine AI, ML, IoT, and cloud technologies?",
        "expected_answer": "IoT sensors collect farm conditions, AI/deep learning is used for crop disease detection, ML supports intelligent agricultural operations, and cloud technologies support the overall platform.",
        "type": "multi_context"
    },
    {
        "question": "What problems in existing systems does the proposed platform attempt to address?",
        "expected_answer": "It addresses limited integration of irrigation and disease detection, limited IoT-AI-mobile integration, irrigation systems that rely only on soil moisture, lack of irrigation history and water analytics, and expensive hardware or subscription requirements.",
        "type": "multi_context"
    },
    {
        "question": "How does the project contribute to sustainable agriculture?",
        "expected_answer": "It supports sustainable agriculture through crop monitoring, early disease detection, intelligent irrigation, reduced unnecessary water use, efficient resource use, and weather-aware irrigation.",
        "type": "multi_context"
    }
]

In [ ]:
evaluation_data += [
    {
        "question": "What is the accuracy of the crop disease detection model?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    },
    {
        "question": "What is the exact amount of water saved by the system per day?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    },
    {
        "question": "What is the exact cost of building the complete system?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    },
    {
        "question": "How many crops can the disease detection model identify?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    },
    {
        "question": "What is the exact irrigation threshold value for soil moisture?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    }
]

In [ ]:
import pandas as pd

evaluation_data = [
    {
        "question": "What is the main project proposed in the synopsis?",
        "expected_answer": "An AI and ML enabled IoT-based precision agriculture platform for smart irrigation and crop disease detection.",
        "type": "direct"
    },
    {
        "question": "Which microcontroller is used in the proposed system?",
        "expected_answer": "ESP32.",
        "type": "direct"
    },
    {
        "question": "What sensors are used to monitor farm conditions?",
        "expected_answer": "Soil moisture, temperature, humidity, rainfall, and water level sensors.",
        "type": "direct"
    },
    {
        "question": "What is the purpose of the smart irrigation system?",
        "expected_answer": "To automatically manage irrigation based on sensor readings and weather forecast information and optimize water usage.",
        "type": "direct"
    },
    {
        "question": "What does the AI-based crop disease detection module analyze?",
        "expected_answer": "Crop leaf images.",
        "type": "direct"
    },
    {
        "question": "Which technology is used for crop disease detection?",
        "expected_answer": "AI/ML and image processing techniques.",
        "type": "direct"
    },
    {
        "question": "Which weather API is mentioned in the software requirements?",
        "expected_answer": "OpenWeatherMap.",
        "type": "direct"
    },
    {
        "question": "What are the main advantages of the proposed system?",
        "expected_answer": "Automatic irrigation, real-time monitoring, early disease detection, crop growth tracking, water and cost savings, reduced manual effort, remote farm monitoring, and a scalable IoT-AI platform.",
        "type": "direct"
    },
    {
        "question": "What hardware is required for the proposed system?",
        "expected_answer": "ESP32, soil moisture sensor, DHT22 sensor, rain sensor, water level sensor, relay module, mini water pump, breadboard, jumper wires, power supply, and smartphone camera.",
        "type": "direct"
    },
    {
        "question": "What software technologies are listed in the project requirements?",
        "expected_answer": "Arduino IDE, Python, Visual Studio Code, Flask/Streamlit, TensorFlow/Keras, OpenCV, Firebase or Supabase, Git/GitHub, OpenWeatherMap, and related tools.",
        "type": "direct"
    },

    {
        "question": "How does the proposed system use IoT and weather information for irrigation?",
        "expected_answer": "The ESP32 collects sensor data and the system uses it together with weather forecast information to automatically manage irrigation.",
        "type": "multi_context"
    },
    {
        "question": "How does the proposed platform help farmers monitor their farms?",
        "expected_answer": "It provides real-time sensor data, irrigation status, weather updates, notifications, and irrigation history.",
        "type": "multi_context"
    },
    {
        "question": "How does the project combine AI, ML, IoT, and cloud technologies?",
        "expected_answer": "IoT sensors collect farm data, AI/deep learning performs disease detection, ML supports intelligent agricultural operations, and cloud technologies support the platform.",
        "type": "multi_context"
    },
    {
        "question": "What problems in existing systems does the proposed platform attempt to address?",
        "expected_answer": "Limited integration of irrigation and disease detection, limited IoT-AI-mobile integration, reliance only on soil moisture, lack of irrigation history and water analytics, and expensive hardware or subscriptions.",
        "type": "multi_context"
    },
    {
        "question": "How does the project contribute to sustainable agriculture?",
        "expected_answer": "It supports sustainable agriculture through crop monitoring, disease detection, intelligent irrigation, reduced water use, and efficient resource management.",
        "type": "multi_context"
    },

    {
        "question": "What is the accuracy of the crop disease detection model?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    },
    {
        "question": "What is the exact amount of water saved by the system per day?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    },
    {
        "question": "What is the exact cost of building the complete system?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    },
    {
        "question": "How many crops can the disease detection model identify?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    },
    {
        "question": "What is the exact irrigation threshold value for soil moisture?",
        "expected_answer": "Not specified in the provided document.",
        "type": "unanswerable"
    }
]

print("Total evaluation questions:", len(evaluation_data))

Total evaluation questions: 20


In [ ]:
results = []

for item in evaluation_data:

    answer, sources, retrieved = rag_answer(item["question"])

    context = "\n\n".join(
        r["text"] for r in retrieved
    )

    results.append({
        "question": item["question"],
        "type": item["type"],
        "expected_answer": item["expected_answer"],
        "rag_answer": answer,
        "retrieved_context": context,
        "sources": sources
    })

evaluation_df = pd.DataFrame(results)

evaluation_df.head()

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

,question,type,expected_answer,rag_answer,retrieved_context,sources
0,What is the main project proposed in the synop...,direct,An AI and ML enabled IoT-based precision agric...,The main project proposed in the synopsis is a...,"• To provide irrigation history, smart notific...","[2, 3, 5, 6]"
1,Which microcontroller is used in the proposed ...,direct,ESP32.,The proposed system uses an ESP32 microcontrol...,ater Level Sensor \n• Relay Module \n• Mini ...,"[1, 2, 4]"
2,What sensors are used to monitor farm conditions?,direct,"Soil moisture, temperature, humidity, rainfall...",Sensors used to monitor farm conditions includ...,ater Level Sensor \n• Relay Module \n• Mini ...,"[1, 2, 4]"
3,What is the purpose of the smart irrigation sy...,direct,To automatically manage irrigation based on se...,The purpose of the proposed smart irrigation s...,"semin Bala, ""IoT and ML-based automatic irriga...","[1, 4]"
4,What does the AI-based crop disease detection ...,direct,Crop leaf images.,The AI-based crop disease detection module ana...,"t informati on, the system automatically \nman...","[1, 3, 4, 5]"


In [ ]:
import json
import time

def evaluate_faithfulness(question, context, answer):

    prompt = f"""
You are evaluating a RAG system.

Question:
{question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Determine whether the Generated Answer is fully supported by the Retrieved Context.

Rules:
1. If the answer is directly supported by the context, mark it as FAITHFUL.
2. If the answer contains information not supported by the context, mark it as HALLUCINATION.
3. If the context does not contain enough information and the answer says that the information is unavailable, mark it as FAITHFUL.
4. Do not use outside knowledge.

Return ONLY this format:

LABEL: FAITHFUL
REASON: short explanation

or

LABEL: HALLUCINATION
REASON: short explanation
"""

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=100,
        temperature=0.0
    )

    return response.choices[0].message.content.strip()

In [ ]:
def evaluate_faithfulness(question, context, answer):

    prompt = f"""
You are evaluating a RAG system.

Question:
{question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Determine whether the Generated Answer is supported by the Retrieved Context.

Rules:
1. If the answer is supported by the context, label it FAITHFUL.
2. If the answer contains information not supported by the context, label it HALLUCINATION.
3. If the context does not contain the answer and the generated answer says the information is unavailable, label it FAITHFUL.
4. Do not use outside knowledge.

Return exactly:

LABEL: FAITHFUL
REASON: explanation

or

LABEL: HALLUCINATION
REASON: explanation
"""

    result = llm_pipeline(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )

    generated_text = result[0]["generated_text"]

    return generated_text

In [ ]:
test = evaluation_df.iloc[0]

result = evaluate_faithfulness(
    test["question"],
    test["retrieved_context"],
    test["rag_answer"]
)

print(result)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are evaluating a RAG system.

Question:
What is the main project proposed in the synopsis?

Retrieved Context:
• To provide irrigation history, smart notifications, and water usage analytics.  
• To reduce water wastage, manual effort, and improve crop productivity through precision agriculture 
Software Requirements – 
• Arduino IDE  
• Arduino Studio 
• Python 3.11+  
• Visual Studio Code  
• Flask / Streamlit  
• HTML, CSS, JavaScript  
• TensorFlow / Keras  
• OpenCV  
• Firebase or Supabase  
• Git & GitHub  
• Weather API (OpenWeatherMap) 
• FastAPI (optional)  
• Chart.js / Recharts (Dashboard graphs) 
 
   
Hardware Requirements –  
• ESP32 Development Board  
• Soil Moisture Sensor  
• DHT22 Temperature & Humidity Sensor  
• Rain Sensor  
• Water Level Sensor  
• Relay Module  
• Mini Water Pump  
• Breadboard  
• Jumper Wires  
• Power Supply  
• Smartphone Camera (Primary), ESP32-CAM (Op

he proposed

secondary connection. 
Advantages 
• Automatic irrigation  
• Real-ti

In [ ]:
def evaluate_faithfulness(question, context, answer):

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
{answer}

Is the Answer completely supported by the Context?

Answer only YES or NO.
"""

    result = llm_pipeline(
        prompt,
        max_new_tokens=5,
        do_sample=False
    )

    generated = result[0]["generated_text"]

    response_only = generated[len(prompt):].strip()

    return response_only

In [ ]:
test = evaluation_df.iloc[0]

result = evaluate_faithfulness(
    test["question"],
    test["retrieved_context"],
    test["rag_answer"]
)

print(result)

[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes:
The


In [ ]:
unanswerable_df = evaluation_df[
    evaluation_df["type"] == "unanswerable"
][["question", "expected_answer", "rag_answer"]]

pd.set_option("display.max_colwidth", 300)

unanswerable_df

,question,expected_answer,rag_answer
15,What is the accuracy of the crop disease detection model?,Not specified in the provided document.,"According to the given material, the crop disease detection model achieved high classification accuracy of 97% in the literature survey. However, this accuracy may vary depending on the specific dataset used for training and testing."
16,What is the exact amount of water saved by the system per day?,Not specified in the provided document.,"The exact amount of water saved by the system per day is not mentioned in the given text. However, it is stated that the system encourages efficient use of agricultural resources by data-driven irrigation, which can support more resource-efficient farming practices. Therefore, the answer to the ..."
17,What is the exact cost of building the complete system?,Not specified in the provided document.,"The exact cost of building the complete system for the AI & ML-enabled IoT-based precision agriculture platform for smart irrigation and crop disease detection is not mentioned in the given material. However, it is stated that the system will require hardware requirements such as an ESP32 develo..."
18,How many crops can the disease detection model identify?,Not specified in the provided document.,The proposed system can identify up to three different types of crop diseases based on the provided context from a PDF document.
19,What is the exact irrigation threshold value for soil moisture?,Not specified in the provided document.,"The exact irrigation threshold value for soil moisture is not mentioned in the given text. However, the context suggests that the proposed AI & ML Enabled IoT -Based Precision Agriculture Platform integrates IoT-based smart irrigation and AI-powered crop disease detection into a single platform...."
